# 02 – Xây dựng đặc trưng point-in-time & chia tập out-of-time

Công việc 2.4–2.5: đặc trưng hành vi theo thời điểm, kiểm soát rò rỉ nhãn, chia train/test theo thời gian.

> **Quyền riêng tư:** `customer_id` được dựng từ số điện thoại khách hàng. Notebook chỉ hiển thị số liệu tổng hợp hoặc mã đã che (`KH_0908***444`). Không in trực tiếp cột `customer_id` / `bill_phone_number` / `bill_full_name`.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]  # bảng màu đã kiểm định CVD
from backend.ml import feature_engineering as fe
from backend.app.core.customer_store import mask_customer_id  # che số điện thoại khi hiển thị

## 1. Thiết lập out-of-time
`prepare_dataset()` (dùng chung với pipeline) suy ra mốc cắt, tính đặc trưng **point-in-time** cho tập huấn luyện/kiểm tra và kiểm chứng nhãn.

In [2]:
from backend.ml.train_advanced import prepare_dataset
data = prepare_dataset()
print("Mốc cắt (cutoff):", data["cutoff"])
print("Cuối cửa sổ kết quả:", data["horizon_end"], f"→ {(data['horizon_end'] - data['cutoff']).total_seconds() / 86400:.2f} ngày")
print(f"Train: {len(data['y_train']):,} khách, churn {data['y_train'].mean():.2%} | Test: {len(data['y_test']):,} khách, churn {data['y_test'].mean():.2%}")
print(f"Nhãn tự tính khớp tệp OOT của nhóm: {data['label_agreement']:.4%}")

Mốc cắt (cutoff): 2022-11-02 21:32:06
Cuối cửa sổ kết quả: 2022-11-11 10:59:44 → 8.56 ngày
Train: 10,893 khách, churn 96.20% | Test: 2,724 khách, churn 96.22%
Nhãn tự tính khớp tệp OOT của nhóm: 99.9927%


## 2. Kiểm soát rò rỉ nhãn (target leakage)
Dữ liệu xuất ra ghi trạng thái **cuối cùng** của đơn. Đơn đặt trước cutoff nhưng được cập nhật sau cutoff có trạng thái mà tại cutoff **chưa biết**.

In [3]:
pd.Series(data["leakage_audit"]).to_frame("giá trị")

,giá trị
orders_before_cutoff,14875
orders_updated_after_cutoff,3863
share_updated_after_cutoff,0.2597
policy,Trạng thái/thời gian xử lý của các đơn này đượ...


In [4]:
# Minh họa: số đơn thành công theo trạng thái cuối cùng (sai) vs point-in-time (đúng) của tập train
orders, cutoff = data["orders"], data["cutoff"]
before = orders[orders["order_time"] < cutoff]
naive_completed = (before["status_group"] == "Completed").groupby(before["customer_id"]).sum()
pit_completed = data["agg_train"]["completed_orders"]
cmp = pd.DataFrame({"naive": naive_completed.reindex(pit_completed.index).fillna(0), "point_in_time": pit_completed})
print(f"Khách có số đơn thành công bị 'biết trước' (naive > point-in-time): {(cmp['naive'] > cmp['point_in_time']).mean():.1%}")
cmp.describe().round(3)

Khách có số đơn thành công bị 'biết trước' (naive > point-in-time): 23.6%


,naive,point_in_time
count,10893.000,10893.000
mean,0.932,0.691
std,0.448,0.556
min,0.000,0.000
25%,1.000,0.000
50%,1.000,1.000
75%,1.000,1.000
max,6.000,6.000


### Thí nghiệm: dùng trạng thái cuối cùng thay vì point-in-time
Huấn luyện cùng một mô hình trên (a) đặc trưng point-in-time và (b) đặc trưng tính từ trạng thái **cuối cùng** của đơn (chứa thông tin phát sinh sau cutoff). Phiên bản (b) không hợp lệ dù kết quả ra sao: khi dự đoán thật, trạng thái cuối của các đơn đang giao chưa tồn tại.

In [5]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

def leaky_features(agg, ids):
    """Thay các trường trạng thái bằng trạng thái CUỐI CÙNG (bao gồm cả thông tin sau cutoff) — chỉ để minh họa."""
    b = before[before["customer_id"].isin(ids)]
    g = b.groupby("customer_id")["status_group"]
    a = agg.copy()
    a["completed_orders"] = g.apply(lambda s: (s == "Completed").sum()).reindex(a.index).fillna(0)
    a["returned_orders"] = g.apply(lambda s: (s == "Returned").sum()).reindex(a.index).fillna(0)
    a["cancelled_orders"] = g.apply(lambda s: (s == "Cancelled").sum()).reindex(a.index).fillna(0)
    return fe.build_model_features(a, data["processing_hours_fill"])

params = dict(n_estimators=200, max_depth=2, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
              scale_pos_weight=(1 - data["y_train"]).sum() / data["y_train"].sum(), random_state=42, n_jobs=1)
res = {}
for name, (Xtr, Xte) in {
    "Point-in-time (đúng)": (data["X_train"], data["X_test"]),
    "Trạng thái cuối cùng (rò rỉ)": (leaky_features(data["agg_train"], data["agg_train"].index), leaky_features(data["agg_test"], data["agg_test"].index)),
}.items():
    m = XGBClassifier(**params).fit(Xtr, data["y_train"])
    res[name] = roc_auc_score(data["y_test"], m.predict_proba(Xte)[:, 1])
out = pd.Series(res, name="ROC-AUC test").round(4).to_frame()
print(f"Chênh lệch do rò rỉ: {res['Trạng thái cuối cùng (rò rỉ)'] - res['Point-in-time (đúng)']:+.4f} ROC-AUC")
out

Chênh lệch do rò rỉ: +0.0078 ROC-AUC


,ROC-AUC test
Point-in-time (đúng),0.6103
Trạng thái cuối cùng (rò rỉ),0.6181


**Nhận xét:** trên bộ dữ liệu này, dùng trạng thái cuối cùng chỉ làm ROC-AUC tăng nhẹ, vì nhãn churn phụ thuộc vào việc khách có phát sinh đơn **mới** hay không, chứ không phụ thuộc trực tiếp vào trạng thái các đơn cũ. Dù vậy, khoảng 1/4 số khách bị 'biết trước' số đơn thành công (ô phía trên), nên đặc trưng point-in-time vẫn bắt buộc để kết quả đánh giá phản ánh đúng điều kiện triển khai.

## 3. Danh sách đặc trưng mô hình

In [6]:
desc = data["X_train"].describe().T[["mean", "std", "min", "50%", "max"]]
desc.insert(0, "tên tiếng Việt", [fe.FEATURE_LABELS.get(f, f) for f in desc.index])
print(len(fe.MODEL_FEATURES), "đặc trưng")
desc.round(3)

33 đặc trưng


,tên tiếng Việt,mean,std,min,50%,max
recency_days,Số ngày từ đơn gần nhất (Recency),11.818,8.287,0.000,12.273,144.158
total_orders,Tổng số đơn (Frequency),1.092,0.343,1.000,1.000,9.000
monetary_log,Giá trị đơn thành công (Monetary),8.207,6.052,0.000,12.495,15.132
avg_order_value_log,Giá trị đơn trung bình,8.179,6.030,0.000,12.468,15.132
customer_age_days,Số ngày từ đơn đầu tiên,12.318,8.315,0.003,12.984,144.158
tenure_days,Khoảng cách đơn đầu – đơn cuối,0.500,2.407,0.000,0.000,47.180
mean_interpurchase_days,Chu kỳ mua lại trung bình,0.455,2.237,0.000,0.000,47.180
is_repeat_customer,Khách đã mua lại,0.080,0.271,0.000,0.000,1.000
orders_last_7d,Số đơn 7 ngày gần nhất,0.372,0.522,0.000,0.000,3.000
orders_last_30d,Số đơn 30 ngày gần nhất,1.077,0.365,0.000,1.000,9.000


In [7]:
# Khác biệt trung bình giữa khách churn và khách mua lại (tập train)
X, y = data["X_train"], data["y_train"]
diff = pd.DataFrame({"churn": X[y == 1].mean(), "mua_lai": X[y == 0].mean()})
diff["chênh lệch chuẩn hóa"] = (diff["mua_lai"] - diff["churn"]) / X.std().replace(0, np.nan)
diff.reindex(diff["chênh lệch chuẩn hóa"].abs().sort_values(ascending=False).index).head(12).round(3)

,churn,mua_lai,chênh lệch chuẩn hóa
orders_last_7d,0.362,0.626,0.506
pending_rate,0.270,0.480,0.478
processing_hours_known,0.743,0.543,-0.453
recency_days,11.948,8.535,-0.412
completed_rate,0.639,0.463,-0.372
customer_age_days,12.433,9.398,-0.365
orders_last_30d,1.072,1.198,0.346
total_orders,1.088,1.200,0.329
avg_order_value_log,8.254,6.275,-0.328
monetary_log,8.281,6.321,-0.324
